In [26]:
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    PreTrainedTokenizerBase
)
import torch
import json
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any
from tqdm.auto import tqdm

JsonObject = dict[str, Any]
ModelFeature = dict[str, list[int]]
Offsets = list[tuple[int, int]]
Window = tuple[int, ModelFeature, Offsets]


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

USE_AMP = DEVICE.type == "cuda"

ENTITY_LABELS = ("ORG", "NAME", "GEO")
ENTITY_CHARS = frozenset("'’ʻʼ‘`-")
TAGS = ("O", "B-ORG", "I-ORG", "B-NAME", "I-NAME", "B-GEO", "I-GEO")
TAG_TO_ID = {tag: i for i, tag in enumerate(TAGS)}
ID_TO_TAG = {i: tag for tag, i in TAG_TO_ID.items()}
MODEL_NAME = "dashakoryakovskaya/uzbek-ner"

def lexicon_label(
    surface: str,
    lexicon: dict[str, Counter[str]],
    *,
    min_support: int,
    min_purity: float,
) -> str | None:
    counts = lexicon.get(surface)
    if not counts:
        return None
    label, support = counts.most_common(1)[0]
    total = sum(counts.values())
    if support < min_support or support / total < min_purity:
        return None
    return label

def tokenize_windows(
    tokenizer: PreTrainedTokenizerBase,
    text: str,
    *,
    max_length: int,
    stride: int,
) -> list[tuple[ModelFeature, Offsets]]:
    """Разбивает текст на перекрывающиеся окна и сохраняет координаты токенов."""

    content_length = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if content_length < 1:
        raise ValueError("max-length is too small for tokenizer special tokens")
    if not 0 <= stride < content_length:
        raise ValueError(f"stride must be between 0 and {content_length - 1}")

    # Сначала токенизируем весь текст без truncation, затем вручную строим
    # окна. Это не позволяет fast tokenizer потерять неполное последнее окно.
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
        verbose=False,
    )
    input_ids = [int(token_id) for token_id in encoded["input_ids"]]
    offsets = [(int(start), int(end)) for start, end in encoded["offset_mapping"]]
    if len(input_ids) != len(offsets):
        raise RuntimeError("tokenizer returned different input_ids and offset_mapping lengths")

    windows: list[tuple[ModelFeature, Offsets]] = []
    step = content_length - stride
    window_starts = range(0, len(input_ids), step) if input_ids else (0,)

    for window_start in window_starts:
        window_end = min(window_start + content_length, len(input_ids))
        content_ids = input_ids[window_start:window_end]
        content_offsets = offsets[window_start:window_end]

        prefix_id = (
            tokenizer.cls_token_id
            if tokenizer.cls_token_id is not None
            else tokenizer.bos_token_id
        )
        suffix_id = (
            tokenizer.sep_token_id
            if tokenizer.sep_token_id is not None
            else tokenizer.eos_token_id
        )
        prefix = [int(prefix_id)] if prefix_id is not None else []
        suffix = [int(suffix_id)] if suffix_id is not None else []
        if len(prefix) + len(suffix) != tokenizer.num_special_tokens_to_add(pair=False):
            raise ValueError("unsupported single-sequence special-token layout")

        window_input_ids = prefix + content_ids + suffix
        window_offsets = (
            [(0, 0)] * len(prefix)
            + content_offsets
            + [(0, 0)] * len(suffix)
        )
        if len(window_input_ids) != len(window_offsets):
            raise RuntimeError("window input_ids and offset_mapping lengths differ")

        feature: ModelFeature = {
            "input_ids": window_input_ids,
            "attention_mask": [1] * len(window_input_ids),
        }
        if "token_type_ids" in tokenizer.model_input_names:
            feature["token_type_ids"] = [0] * len(window_input_ids)
        windows.append((feature, window_offsets))

        if window_end >= len(input_ids):
            break
    return windows

def expand_to_attached_word(text: str, start: int, end: int) -> tuple[int, int]:
    """Expands a span only inside the same whitespace-free alphanumeric word."""

    while (
        start > 0
        and start < len(text)
        and is_entity_char(text[start - 1])
        and is_entity_char(text[start])
    ):
        start -= 1
    while (
        end > 0
        and end < len(text)
        and is_entity_char(text[end - 1])
        and is_entity_char(text[end])
    ):
        end += 1
    return start, end

def _build_windows(
    records: list[JsonObject],
    tokenizer: Any,
    *,
    max_length: int,
    stride: int,
) -> list[Window]:
    """Токенизирует все документы и связывает окна с индексами записей."""

    windows: list[Window] = []
    for record_index, record in enumerate(tqdm(records, desc="Tokenize", unit="doc")):
        for feature, offsets in tokenize_windows(
            tokenizer,
            record["text"],
            max_length=max_length,
            stride=stride,
        ):
            windows.append((record_index, feature, offsets))
    return windows

def decode_bio_tokens(tagged_tokens: list[tuple[int, int, str]]) -> list[dict[str, Any]]:
    entities, current = [], None
    for start, end, tag in tagged_tokens:
        if tag == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or current is None or current["label"] != label:
            if current is not None:
                entities.append(current)
            current = {"label": label, "start": start, "end": end}
        else:
            current["end"] = max(current["end"], end)
    if current is not None:
        entities.append(current)
    unique = {(e["label"], e["start"], e["end"]): e for e in entities}
    return sorted(unique.values(), key=lambda e: (e["start"], e["end"], e["label"]))

@torch.inference_mode()
def predict_records(
    model: torch.nn.Module,
    tokenizer: PreTrainedTokenizerBase,
    records: list[dict[str, Any]],
    windows,
) -> list[dict[str, Any]]:
    model.eval()
    aggregated: list[dict[tuple[int, int], tuple[torch.Tensor, int]]] = [
        {} for _ in records
    ]
    for batch_start in tqdm(
        range(0, len(windows), 16),
        desc="Exact-span predict",
        unit="batch",
    ):
        batch_windows = windows[batch_start:batch_start + 16]
        padded = tokenizer.pad(
            [feature for _, feature, _ in batch_windows],
            padding=True,
            return_tensors="pt",
        )
        padded = {key: value.to(DEVICE) for key, value in padded.items()}
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(**padded).logits
        probabilities = torch.softmax(logits.float(), dim=-1).cpu()

        for row, (record_index, _, offsets) in enumerate(batch_windows):
            record_scores = aggregated[record_index]
            for token_index, (start, end) in enumerate(offsets):
                if start == end:
                    continue
                score = probabilities[row, token_index]
                if (start, end) in record_scores:
                    previous, count = record_scores[(start, end)]
                    record_scores[(start, end)] = (previous + score, count + 1)
                else:
                    record_scores[(start, end)] = (score.clone(), 1)

    output = []
    for record, scores in zip(records, aggregated):
        tagged = []
        for (start, end), (score_sum, count) in sorted(scores.items()):
            tag_id = int((score_sum / count).argmax().item())
            tagged.append((start, end, ID_TO_TAG[tag_id]))
        output.append({
            "hash": record["hash"],
            "entities": decode_bio_tokens(tagged),
        })
    return output

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_jsonl(path: Path) -> list[JsonObject]:
    """Читает непустой JSONL и проверяет уникальность hash."""

    records: list[JsonObject] = []
    seen_hashes: set[str] = set()
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                raise ValueError(f"{path}:{line_number}: empty line")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: invalid JSON: {error}") from error
            if not isinstance(record, dict):
                raise ValueError(f"{path}:{line_number}: record must be an object")
            record_hash = record.get("hash")
            if not isinstance(record_hash, str) or not record_hash:
                raise ValueError(f"{path}:{line_number}: hash must be a non-empty string")
            if record_hash in seen_hashes:
                raise ValueError(f"{path}:{line_number}: duplicate hash {record_hash}")
            seen_hashes.add(record_hash)
            records.append(record)
    if not records:
        raise ValueError(f"{path}: no records")
    return records


APOSTROPHES = "'’ʻʼ‘`"

def is_entity_char(char: str) -> bool:
    """Characters allowed inside a conservatively expanded entity word."""

    return char.isalnum() or char in ENTITY_CHARS


def normalize_surface(surface: str) -> str:
    """Нормализует регистр, пробелы и совместимые символы апострофа."""

    normalized = surface.casefold()
    for apostrophe in APOSTROPHES:
        normalized = normalized.replace(apostrophe, "'")
    return " ".join(normalized.split())


def build_normalized_lexicon(
    exact_lexicon: dict[str, Counter[str]],
) -> dict[str, Counter[str]]:
    """Объединяет train-статистику вариантов регистра и апострофов."""

    result: dict[str, Counter[str]] = defaultdict(Counter)
    for surface, counts in exact_lexicon.items():
        result[normalize_surface(surface)].update(counts)
    return dict(result)


def confident_label(
    counts: Counter[str] | None,
    *,
    min_support: int,
    min_purity: float,
) -> tuple[str, int] | None:
    """Возвращает класс и support только для достаточно чистой записи."""

    if not counts:
        return None
    label, support = counts.most_common(1)[0]
    if support < min_support or support / sum(counts.values()) < min_purity:
        return None
    return label, support


def normalized_relabel(
    input_records: list[JsonObject],
    predictions: list[JsonObject],
    normalized_lexicon: dict[str, Counter[str]],
    *,
    min_support: int,
    min_purity: float,
) -> tuple[list[JsonObject], Counter[str]]:
    """Исправляет класс по train-лексикону после нормализации поверхности."""

    texts = {record["hash"]: record["text"] for record in input_records}
    result: list[JsonObject] = []
    stats: Counter[str] = Counter()

    for record in predictions:
        text = texts[record["hash"]]
        corrected: list[JsonObject] = []
        for entity in record["entities"]:
            start, end = entity["start"], entity["end"]
            label = entity["label"]
            decision = confident_label(
                normalized_lexicon.get(normalize_surface(text[start:end])),
                min_support=min_support,
                min_purity=min_purity,
            )
            if decision is not None and decision[0] != label:
                label = decision[0]
                stats["normalized_label_changes"] += 1
            corrected.append({"label": label, "start": start, "end": end})

        unique = {
            (entity["label"], entity["start"], entity["end"]): entity
            for entity in corrected
        }
        stats["duplicates_removed"] += len(corrected) - len(unique)
        result.append({"hash": record["hash"], "entities": list(unique.values())})
    return result, stats


def has_entity_boundaries(text: str, start: int, end: int) -> bool:
    """Не позволяет находить короткую поверхность внутри большего слова."""

    left_is_inside_word = (
        start > 0
        and is_entity_char(text[start - 1])
        and is_entity_char(text[start])
    )
    right_is_inside_word = (
        end < len(text)
        and is_entity_char(text[end - 1])
        and is_entity_char(text[end])
    )
    return not left_is_inside_word and not right_is_inside_word


def propagate_repeated_mentions(
    input_records: list[JsonObject],
    predictions: list[JsonObject],
    *,
    min_length: int,
) -> tuple[list[JsonObject], Counter[str]]:
    """Переносит распознанный exact surface на другие упоминания в документе."""

    texts = {record["hash"]: record["text"] for record in input_records}
    result: list[JsonObject] = []
    stats: Counter[str] = Counter()

    for record in predictions:
        text = texts[record["hash"]]
        entities = list(record["entities"])
        seeds = list(entities)
        occupied = [(entity["start"], entity["end"]) for entity in entities]

        for seed in seeds:
            surface = text[seed["start"] : seed["end"]]
            if len(surface) < min_length:
                continue

            search_from = 0
            while True:
                start = text.find(surface, search_from)
                if start < 0:
                    break
                end = start + len(surface)
                search_from = start + 1

                if not has_entity_boundaries(text, start, end):
                    continue
                if any(start < old_end and end > old_start for old_start, old_end in occupied):
                    continue

                entities.append(
                    {"label": seed["label"], "start": start, "end": end}
                )
                occupied.append((start, end))
                stats["repeated_mentions_added"] += 1

        result.append({"hash": record["hash"], "entities": entities})
    return result, stats


def resolve_overlaps(
    input_records: list[JsonObject],
    predictions: list[JsonObject],
    normalized_lexicon: dict[str, Counter[str]],
) -> tuple[list[JsonObject], Counter[str]]:
    """Оставляет один span из пересекающейся группы, предпочитая длинный."""

    texts = {record["hash"]: record["text"] for record in input_records}
    result: list[JsonObject] = []
    stats: Counter[str] = Counter()

    for record in predictions:
        text = texts[record["hash"]]

        def rank(entity: JsonObject) -> tuple[int, int, int, int]:
            surface = text[entity["start"] : entity["end"]]
            counts = normalized_lexicon.get(normalize_surface(surface))
            decision = confident_label(counts, min_support=2, min_purity=0.9)
            known = int(decision is not None and decision[0] == entity["label"])
            support = decision[1] if known else 0
            return (len(surface), known, support, -entity["start"])

        kept: list[JsonObject] = []
        for entity in sorted(record["entities"], key=rank, reverse=True):
            overlaps = any(
                entity["start"] < previous["end"]
                and entity["end"] > previous["start"]
                for previous in kept
            )
            if overlaps:
                stats["overlapping_entities_removed"] += 1
            else:
                kept.append(entity)
        result.append({"hash": record["hash"], "entities": kept})
    return result, stats


def filter_short_unknown_entities(
    input_records: list[JsonObject],
    predictions: list[JsonObject],
    exact_lexicon: dict[str, Counter[str]],
    *,
    max_length: int,
) -> tuple[list[JsonObject], Counter[str]]:
    """Удаляет неизвестные train-лексикону spans длиной не более N символов."""

    texts = {record["hash"]: record["text"] for record in input_records}
    result: list[JsonObject] = []
    stats: Counter[str] = Counter()

    for record in predictions:
        text = texts[record["hash"]]
        kept: list[JsonObject] = []
        for entity in record["entities"]:
            surface = text[entity["start"] : entity["end"]]
            known_with_label = entity["label"] in exact_lexicon.get(surface, {})
            if len(surface.strip()) <= max_length and not known_with_label:
                stats["short_unknown_entities_removed"] += 1
                continue
            kept.append(entity)
        result.append({"hash": record["hash"], "entities": kept})
    return result, stats


def sort_and_deduplicate(predictions: list[JsonObject]) -> list[JsonObject]:
    result: list[JsonObject] = []
    for record in predictions:
        unique = {
            (entity["label"], entity["start"], entity["end"]): entity
            for entity in record["entities"]
        }
        result.append(
            {
                "hash": record["hash"],
                "entities": sorted(
                    unique.values(),
                    key=lambda entity: (
                        entity["start"],
                        entity["end"],
                        entity["label"],
                    ),
                ),
            }
        )
    return result

def postprocess_predictions(
    input_records: list[JsonObject],
    predictions: list[JsonObject],
    lexicon: dict[str, Counter[str]],
    *,
    expand_word_boundaries: bool = True,
    relabel: bool = True,
    min_label_support: int = 2,
    min_label_purity: float = 0.9,
) -> tuple[list[JsonObject], Counter[str]]:
    """Applies train-only boundary and label corrections."""

    texts: dict[str, str] = {}
    for record in input_records:
        record_hash = record.get("hash")
        text = record.get("text")
        if not isinstance(record_hash, str) or not isinstance(text, str):
            raise ValueError("input records must contain string hash and text")
        if record_hash in texts:
            raise ValueError(f"duplicate input hash: {record_hash}")
        texts[record_hash] = text

    prediction_hashes = {record.get("hash") for record in predictions}
    if prediction_hashes != set(texts):
        raise ValueError("input and prediction hash sets differ")

    result: list[JsonObject] = []
    stats: Counter[str] = Counter()
    for record in predictions:
        record_hash = record["hash"]
        text = texts[record_hash]
        entities = record.get("entities")
        if not isinstance(entities, list):
            raise ValueError(f"prediction {record_hash}: entities must be a list")

        corrected: list[JsonObject] = []
        for entity in entities:
            label = entity.get("label")
            start = entity.get("start")
            end = entity.get("end")
            if (
                label not in ENTITY_LABELS
                or not isinstance(start, int)
                or not isinstance(end, int)
                or not 0 <= start < end <= len(text)
            ):
                raise ValueError(f"prediction {record_hash}: invalid entity {entity!r}")

            if expand_word_boundaries:
                new_start, new_end = expand_to_attached_word(text, start, end)
                if (new_start, new_end) != (start, end):
                    stats["boundary_changes"] += 1
                    start, end = new_start, new_end

            if relabel:
                new_label = lexicon_label(
                    text[start:end],
                    lexicon,
                    min_support=min_label_support,
                    min_purity=min_label_purity,
                )
                if new_label is not None and new_label != label:
                    stats["label_changes"] += 1
                    label = new_label
            corrected.append({"label": label, "start": start, "end": end})

        unique = {
            (entity["label"], entity["start"], entity["end"]): entity
            for entity in corrected
        }
        stats["duplicates_removed"] += len(corrected) - len(unique)
        result.append(
            {
                "hash": record_hash,
                "entities": sorted(
                    unique.values(),
                    key=lambda entity: (entity["start"], entity["end"], entity["label"]),
                ),
            }
        )
    return result, stats


def build_surface_lexicon(
    train_records: list[JsonObject],
) -> dict[str, Counter[str]]:
    """Counts exact, case-sensitive entity surfaces by label in train."""

    lexicon: dict[str, Counter[str]] = defaultdict(Counter)
    for record in train_records:
        text = record.get("text")
        entities = record.get("entities")
        if not isinstance(text, str) or not isinstance(entities, list):
            raise ValueError("train records must contain text and entities")
        for entity in entities:
            label = entity.get("label")
            start = entity.get("start")
            end = entity.get("end")
            if label not in ENTITY_LABELS or not isinstance(start, int) or not isinstance(end, int):
                raise ValueError("invalid train entity")
            lexicon[text[start:end]][label] += 1
    return dict(lexicon)


def postprocess(input_records, predictions, train_path):
    train_records = read_jsonl(train_path)

    exact_lexicon = build_surface_lexicon(train_records)
    normalized_lexicon = build_normalized_lexicon(exact_lexicon)
    result, stats = postprocess_predictions(
        input_records,
        predictions,
        exact_lexicon,
        expand_word_boundaries=True,
        relabel=True,
        min_label_support=2,
        min_label_purity=0.9,
    )

    result, stage_stats = normalized_relabel(
        input_records,
        result,
        normalized_lexicon,
        min_support=2,
        min_purity=1.0,
    )
    stats.update(stage_stats)

    result, stage_stats = propagate_repeated_mentions(
        input_records,
        result,
        min_length=4,
    )
    stats.update(stage_stats)

    result, stage_stats = resolve_overlaps(
        input_records,
        result,
        normalized_lexicon,
    )
    stats.update(stage_stats)

    result, stage_stats = filter_short_unknown_entities(
        input_records,
        result,
        exact_lexicon,
        max_length=2,
    )
    stats.update(stage_stats)

    result = sort_and_deduplicate(result)

    print(f"Records: {len(result)}")
    for name in (
        "boundary_changes",
        "label_changes",
        "normalized_label_changes",
        "repeated_mentions_added",
        "overlapping_entities_removed",
        "short_unknown_entities_removed",
        "duplicates_removed",
    ):
        print(f"{name}: {stats[name]}")
    return result



def predict(records, train_path):
    tokenizer = AutoTokenizer.from_pretrained(
        "FacebookAI/xlm-roberta-large",
        use_fast=True,
    )
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(TAGS),
        id2label=ID_TO_TAG,
        label2id=TAG_TO_ID,
    ).to(DEVICE)
    windows = _build_windows(
        records, tokenizer, max_length=512, stride=128
    )
    predictions = predict_records(
                model,
                tokenizer,
                records,
                windows,
            )
    return postprocess(records, predictions, train_path)

In [27]:
TEST_PATH = Path('/content/public_test_inputs.jsonl')
TRAIN_PATH = Path('/content/train.jsonl')
test_records = read_jsonl(TEST_PATH)
predict(test_records, TRAIN_PATH)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Tokenize:   0%|          | 0/1000 [00:00<?, ?doc/s]

Exact-span predict:   0%|          | 0/74 [00:00<?, ?batch/s]

Records: 1000
boundary_changes: 228
label_changes: 3
normalized_label_changes: 0
repeated_mentions_added: 40
overlapping_entities_removed: 56
short_unknown_entities_removed: 10
duplicates_removed: 91


[{'hash': '00007f7fa410ad6765669932bdf9cf7b20260308', 'entities': []},
 {'hash': '000298899f08a4d8ab4bbfd625598ec820260311',
  'entities': [{'label': 'ORG', 'start': 89, 'end': 92}]},
 {'hash': '00072b18f0595d8014a03caf300b12b020260304', 'entities': []},
 {'hash': '000b53d4f20ba68cdb6957b9bc3ca66520260227', 'entities': []},
 {'hash': '000de1b2d89d1aca4855c2314e5f0d5620260309', 'entities': []},
 {'hash': '000e629a663ffef971a7810a2179d76820260313',
  'entities': [{'label': 'ORG', 'start': 0, 'end': 16},
   {'label': 'ORG', 'start': 35, 'end': 43}]},
 {'hash': '0010548b3695cf482ea9f3e98753a31520260312', 'entities': []},
 {'hash': '0014c38c4a95a7b7c4618bac66d0e6c420260529',
  'entities': [{'label': 'ORG', 'start': 0, 'end': 3},
   {'label': 'ORG', 'start': 182, 'end': 185},
   {'label': 'GEO', 'start': 205, 'end': 214}]},
 {'hash': '001de0359f82ceb0955666cdadbf3ec520260228',
  'entities': [{'label': 'ORG', 'start': 0, 'end': 13},
   {'label': 'ORG', 'start': 20, 'end': 32},
   {'label': 'O